# ✈️ Orquestador ELT – Flights & Airports

Este notebook ejecuta y documenta el flujo **Extract → Load → Transform (ELT)** del proyecto.

## 🎯 Objetivos
- **Extract:** consultar la API de AviationStack y persistir crudos en **Parquet**.
- **Load:** cargar crudos a **Delta Lake** (control de esquema e histórico).
- **Transform:** aplanar, limpiar y enriquecer datos listos para análisis.

## ✅ Justificación de elecciones
- **Parquet:** formato columnar eficiente para crudos, compresión y tipos estables.
- **Delta Lake:** *ACID*, control de esquema, versionado, y soporte de partición por fecha.
- **Pandas:** flexibilidad y velocidad para prototipado de transformaciones.
- **Notebook:** narrativa reproducible, perfecta para defensa de la consigna.


## 🔧 Configuración inicial

In [1]:
# Asegurar imports y rutas
import os
import pandas as pd
from deltalake import DeltaTable
from pathlib import Path

# Rutas principales del data lake
DATA_LAKE = Path("../data_lake")
FLIGHTS_PROCESSED = DATA_LAKE / "flights" / "processed"
AIRPORTS_PROCESSED = DATA_LAKE / "airports" / "processed"
ENRICHED = DATA_LAKE / "flights" / "processed_enriched_pandas"
AGG = DATA_LAKE / "flights" / "agg_by_status"

DATA_LAKE, FLIGHTS_PROCESSED, AIRPORTS_PROCESSED, ENRICHED, AGG


(WindowsPath('../data_lake'),
 WindowsPath('../data_lake/flights/processed'),
 WindowsPath('../data_lake/airports/processed'),
 WindowsPath('../data_lake/flights/processed_enriched_pandas'),
 WindowsPath('../data_lake/flights/agg_by_status'))

## 1️⃣ Extract

Ejecuta `extract.py`. En caso de error de API (401/429), el script **siempre** guarda Parquet con columnas esperadas, evitando que el pipeline se rompa.


In [2]:
!python ../src/extract.py

C:\Users\MONSO\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'c:\\Users\\MONSO\\src\\extract.py': [Errno 2] No such file or directory


## 2️⃣ Load

Ejecuta `load.py` y escribe tablas Delta. Se utiliza **hard overwrite** en la primera corrida para evitar conflictos de esquema entre ejecuciones de prueba.


In [3]:
!python ../src/load.py

C:\Users\MONSO\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'c:\\Users\\MONSO\\src\\load.py': [Errno 2] No such file or directory


## 3️⃣ Transform

Ejecuta `transform.py` que:
- Aplana estructuras JSON (`departure`, `arrival`, `airline`, `flight`, `aircraft`, `live`).
- Normaliza tipos, maneja nulos y crea columnas faltantes críticas.
- Enriquce con nombres de aeropuertos y marca vuelos con retraso (`is_delayed`).

In [4]:
!python ../src/transform.py

C:\Users\MONSO\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'c:\\Users\\MONSO\\src\\transform.py': [Errno 2] No such file or directory


## 4️⃣ Validación rápida de resultados

Leemos las tablas Delta generadas y mostramos conteos básicos para asegurar que el pipeline corrió.


In [5]:
from utils import log

def load_delta_safe(path: Path):
    if path.exists():
        return DeltaTable(str(path)).to_pandas()
    else:
        log(f"⚠️ No existe la tabla: {path}")
        return pd.DataFrame()

df_flights = load_delta_safe(FLIGHTS_PROCESSED)
df_airports = load_delta_safe(AIRPORTS_PROCESSED)
df_enriched = load_delta_safe(ENRICHED)

log(f"Vuelos (raw→Delta): {len(df_flights)} filas")
log(f"Aeropuertos (raw→Delta): {len(df_airports)} filas")
log(f"Vuelos enriquecidos: {len(df_enriched)} filas")

# Mostrar un preview elegante
display(df_enriched.head(10))


ModuleNotFoundError: No module named 'utils'

## 5️⃣ Métricas y visualizaciones

Algunas métricas útiles para la demo:
- Distribución por `flight_status`.
- % de vuelos con retraso (`is_delayed`).
- Top aeropuertos de salida por volumen.


In [ ]:
import matplotlib.pyplot as plt

if not df_enriched.empty:
    # Distribución por estado
    status_counts = df_enriched['flight_status'].value_counts(dropna=False).sort_values(ascending=False)
    ax = status_counts.plot(kind='bar', figsize=(7,4))
    ax.set_title('Distribución de vuelos por estado')
    ax.set_xlabel('Estado')
    ax.set_ylabel('Cantidad de vuelos')
    plt.show()

    # % retrasados
    if 'is_delayed' in df_enriched.columns:
        delayed_rate = (df_enriched['is_delayed'].astype(bool).mean() * 100)
        print(f"% de vuelos retrasados: {delayed_rate:.2f}%")

    # Top aeropuertos de salida
    dep_col = 'departure_airport_name' if 'departure_airport_name' in df_enriched.columns else 'departure_iata'
    top_airports = df_enriched[dep_col].value_counts().head(10)
    ax = top_airports.plot(kind='bar', figsize=(7,4))
    ax.set_title('Top aeropuertos de salida (Top 10)')
    ax.set_xlabel('Aeropuerto')
    ax.set_ylabel('Vuelos')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos enriquecidos para visualizar.")


---
### ♻️ Reproducibilidad
- Ejecutá las celdas en orden (1→2→3→4→5).
- Si cambiás el esquema en pruebas, corré de nuevo **Load** y **Transform** para regenerar Delta con *hard overwrite*.
- Configurá `AVIATIONSTACK_API_KEY` en tu entorno para obtener datos reales.
